# ITOM 6219 — Week 1: Basics of AI Application

This notebook accompanies the Week 1 lecture notes. You will practice two things:

1. **Prompt Engineering & RAG** — Call the OpenAI API, experiment with prompts, and see how a simple RAG system works.
2. **Web App Basics** — Build a tiny interactive web page right inside this notebook using HTML, JavaScript, and CSS.

Most of the code is already written for you. Your job is to **fill in the blanks** marked with `___`.

---
**Before you start:** Add your OpenAI API key to the Colab sidebar.
- Click the **key icon** on the left panel ("Secrets").
- Click **"Add new secret"**.
- Name it exactly: `OPENAI_API_KEY`
- Paste the key your instructor provided as the value.
- Toggle **"Notebook access"** to on.

## Setup — Run This First

Run the cell below to install the OpenAI library and load your API key. You only need to do this once per session.

In [ ]:
# ── SETUP ──────────────────────────────────────────────────────────────────
# This cell installs the OpenAI Python library so we can talk to the AI model.
# The "-q" flag means "quiet" — it suppresses long installation messages.
!pip install openai -q

# "google.colab" is a helper library built into Google Colab.
# "userdata" lets us safely read the secret API key you stored in the sidebar
# without ever typing it in plain text.
from google.colab import userdata

# "OpenAI" is the main class we use to send requests to the AI model.
from openai import OpenAI

# Retrieve your API key from the Colab Secrets panel (the key icon on the left).
# This key tells OpenAI who you are and authorizes you to use the model.
api_key = userdata.get('OPENAI_API_KEY')

# Create a "client" object — think of it as opening a phone line to the AI.
# Every time we want to talk to the model, we'll use this client.
client = OpenAI(api_key=api_key)

print("Setup complete!")

---
# Part 1: Prompt Engineering & RAG

## 1.1 Your First API Call

Just run this cell — nothing to fill in yet. Read the output and make sure it works.

In [ ]:
# ── 1.1 YOUR FIRST API CALL ────────────────────────────────────────────────
# We ask the model to complete an unfinished sentence.
# client.completions.create() sends a request to the OpenAI API and returns a response.
response = client.completions.create(
    model="gpt-3.5-turbo-instruct",  # The specific AI model we are using
    prompt="It is raining today, and I need to bring my ",  # The unfinished sentence
    max_tokens=10,    # Maximum number of words (roughly) the model can generate
    temperature=0     # 0 = fully deterministic — same input always gives same output
)

# The response object contains a list of "choices".
# choices[0] is the first (and only) result.
# .text gives the raw text; .strip() removes any leading/trailing whitespace.
print(response.choices[0].text.strip())

The model **completed** the sentence based on your prompt. This is called a **completion** model — it predicts what comes next.

---
## 1.2 Exercise: Zero-Shot Prompting

Zero-shot prompting means giving the model an instruction with no examples — just tell it what to do.

**Your task:** Fill in a prompt that asks the model to write a one-sentence product description for a reusable coffee cup.

In [ ]:
# ── 1.2 ZERO-SHOT PROMPTING ────────────────────────────────────────────────
# Replace the three underscores below with your own prompt — no examples needed.
# Example: "Write a one-sentence product description for a reusable coffee cup."
my_prompt = "___"

# Send the prompt to the model and store the response
response = client.completions.create(
    model="gpt-3.5-turbo-instruct",
    prompt=my_prompt,
    max_tokens=60,    # Allow up to ~60 tokens (roughly 45 words) in the output
    temperature=0
)

# Print just the text portion of the response
print(response.choices[0].text.strip())

> **Tip:** A good zero-shot prompt is specific. Compare:
> - Vague: `"Write about a coffee cup."`
> - Specific: `"Write a one-sentence product description for a reusable coffee cup targeting college students."`

Try both versions and observe the difference in output.

---
## 1.3 Exercise: Few-Shot Prompting

Few-shot prompting means giving the model **one or more examples** before your actual question. This helps the model understand the format and style you want.

The prompt below already has one example (EcoPure). Your task: **add a second example** for a standing desk, then let the model generate copy for a noise-canceling headphone.

In [ ]:
# ── 1.3 FEW-SHOT PROMPTING ─────────────────────────────────────────────────
# In few-shot prompting, we give the model examples before asking it to generate.
# The model learns the format and style from the examples, then continues the pattern.
#
# The ### delimiter separates each example — it helps the model know where one ends
# and the next begins. Think of it like a section divider in a form.
#
# YOUR TASK: Fill in the target market and advertising copy for the StandUp Pro Desk.

prompt = """
Use a product name and a target market to create a short advertising copy.

###
product name: EcoPure Hydration Bottle
target market: environmentally conscious consumers
advertising copy: Stay refreshed and make a difference — the last water bottle you will ever need.

###
product name: StandUp Pro Desk
target market: ___
advertising copy: ___

###
product name: FocusSound Headphones
target market: graduate students who study in noisy cafes
advertising copy:
"""

# Send the full prompt (examples + the incomplete final entry) to the model.
# The model will complete the last "advertising copy:" field.
response = client.completions.create(
    model="gpt-3.5-turbo-instruct",
    prompt=prompt,
    max_tokens=80,
    temperature=0
)

print(response.choices[0].text.strip())

> **Notice:** The model picked up the style from the examples and continued the pattern. That is the power of few-shot prompting.

---
## 1.4 Exercise: Hyperparameters — Temperature

**Temperature** controls how creative or random the model's response is.
- `temperature=0` → very predictable, same answer every time
- `temperature=1.5` → creative but may drift

Run the cell below **three times** with `temperature=0`, then change it to `temperature=1.2` and run three more times. What do you notice?

In [ ]:
# ── 1.4 HYPERPARAMETERS — TEMPERATURE ─────────────────────────────────────
# Temperature controls how "creative" the model is.
#   0   → very predictable; same prompt always gives the same answer
#   0.7 → balanced; some variation but still coherent
#   1.2 → creative and surprising; may occasionally drift off-topic
#
# YOUR TASK: Replace ___ with a number (try 0, then 0.7, then 1.2).
# Run this cell multiple times for each value and compare the outputs.
temperature_value = ___

response = client.completions.create(
    model="gpt-3.5-turbo-instruct",
    prompt="Write a catchy tagline for a smartwatch that tracks your mood.",
    max_tokens=30,
    temperature=temperature_value  # The value you set above is passed here
)

# f-strings (f"...") let us embed variables inside a string using curly braces.
# For example, f"Temperature: {temperature_value}" prints "Temperature: 0" if the value is 0.
print(f"Temperature: {temperature_value}")
print(response.choices[0].text.strip())

> **Reflection:** For advertising copy, which temperature range feels most useful? Why?

---
## 1.5 Mini RAG: Answering from a Knowledge Base

RAG stands for **Retrieval-Augmented Generation**. Instead of relying only on the model's training data, we *retrieve* relevant information first, then pass it to the model as context.

Below is a tiny "knowledge base" about SMU Cox MBA programs. The model does not know these specific details — but we can feed them in via the prompt.

**Your task:** Fill in the question and observe how the model answers using the provided context.

In [ ]:
# ── 1.5 MINI RAG: ANSWERING FROM A KNOWLEDGE BASE ─────────────────────────
# RAG = Retrieval-Augmented Generation.
# Instead of relying on what the model already knows (which may be outdated),
# we paste relevant facts directly into the prompt as "context".
# The model then answers using only those facts — reducing hallucination.

# This is our mini "knowledge base" — a block of text with facts about SMU Cox.
# In a real RAG system, this text would be retrieved automatically from a database.
knowledge_base = """
The SMU Cox Full-Time MBA program is a two-year program located in Dallas, Texas.
Students complete a core curriculum in year one and choose a concentration in year two.
Available concentrations include Finance, Marketing, Strategy, and Business Analytics.
The average program size is around 60-70 students.
The program emphasizes real-world projects and has strong ties to Dallas-based companies.
"""

# YOUR TASK: Replace ___ with a question a prospective student might ask.
# Example: "What concentrations are available?"
student_question = "___"

# We build the full prompt by combining the knowledge base and the question.
# The f-string below inserts the variables {knowledge_base} and {student_question}
# into the prompt text at the marked positions.
rag_prompt = f"""
Answer the question below using only the information provided. If the answer is not in the text, say "I don't have that information."

Information:
{knowledge_base}

Question: {student_question}
Answer:
"""

response = client.completions.create(
    model="gpt-3.5-turbo-instruct",
    prompt=rag_prompt,
    max_tokens=80,
    temperature=0   # Keep at 0 for factual Q&A — we want consistent, grounded answers
)

print(response.choices[0].text.strip())

> **Try this:** Ask the model something that is NOT in the knowledge base (e.g., "What is the tuition?"). What does it say?

This is the essence of RAG — the model is grounded by the context you provide, which reduces hallucination.

## 1.6 Alignment (in-context learning)

In [ ]:
# ── 1.6 ALIGNMENT (IN-CONTEXT LEARNING) ───────────────────────────────────
# Here we align the model with a real business metric: click-through rate (CTR).
# Instead of just giving examples of good copy (few-shot), we also show bad copy
# and label each one as GOOD or BAD based on its measured CTR.
# The model learns what "GOOD" looks like — specific, urgent, benefit-driven —
# and avoids the patterns associated with "BAD" copy.
#
# YOUR TASK:
#   1. Fill in ___ for each CTR label: write GOOD or BAD based on the CTR value.
#   2. Fill in a product name and target market at the bottom for the model to generate.

prompt = """
Use past promotional copies and their click-through rates (CTR) to write a new high-performing promotional copy.
GOOD means the copy had a high CTR. BAD means the copy had a low CTR.

###
promotional copy: Stay hydrated in style — EcoPure keeps drinks cold for 48 hours. Shop now and enjoy 20% off today!
CTR: 8.5% — ___

###
promotional copy: The last bottle you'll ever need. Eco-friendly. BPA-free. 48-hour cold. Get yours before it sells out.
CTR: 6.2% — ___

###
promotional copy: EcoPure Hydration Bottle is a sustainable product made from eco-friendly materials. It keeps your drinks cold for a long time.
CTR: 1.1% — ___

###
promotional copy: Buy our water bottle. It is a good bottle for drinking water. Available online.
CTR: 0.8% — ___

###
product name: ___
target market: ___
promotional copy:
"""

# temperature=0.7 gives a little creativity — useful for ad copy,
# while still staying close to the GOOD examples in the prompt.
response = client.completions.create(
    model="gpt-3.5-turbo-instruct",
    prompt=prompt,
    max_tokens=80,
    temperature=0.7
)

print(response.choices[0].text.strip())

> **Reflection questions:**
> 1. Remove the `GOOD`/`BAD` labels and re-run — does the output change? Why?
> 2. Try keeping only the two GOOD examples (standard few-shot) vs. all four with labels (alignment). Which gives better results?
> 3. In your AI app, how could you use real CTR data from Google Analytics to continuously improve your prompt?

This is a lightweight form of **alignment**: steering the model toward outputs that match real-world success metrics, no retraining required.

This approach is supported by recent research. **URIAL** (Lin et al., ICLR 2024) demonstrates that alignment can be achieved purely through in-context learning — using a handful of stylistic, labeled examples in the prompt — matching the performance of models fine-tuned with full RLHF. Similarly, **ICPL** (Yu et al., 2024) shows that LLMs have native preference-learning capabilities: embedding outcome-labeled examples (such as CTR-tagged copy) directly in the prompt enables sample-efficient alignment without any model retraining.

---
## 1.7 Function Calling (Not required for the class)

So far, the model has only known what it was trained on. **Function calling** lets the model request real-world data by asking your code to run a function and return the result.

How it works in three steps:
1. You define a function (e.g., `get_weather`) and tell the model it exists.
2. The model decides to call it and returns structured JSON with the arguments (e.g., `{"city": "Dallas"}`).
3. Your code runs the function and sends the result back — the model uses it to answer naturally.

**Your task:** Run the cell below and observe the two-step process in the output.

In [ ]:
# ── 1.7 FUNCTION CALLING ───────────────────────────────────────────────────
# Function calling uses a chat model (gpt-4o-mini), not a completion model.
# Chat models support "tools" — a list of functions the model can choose to call.
import json

# ── STEP 1: Define the function the model is allowed to call ────────────────
# This is a simulated weather function — it returns fake data.
# In a real app, you would call a live weather API (e.g., OpenWeatherMap) here.
def get_weather(city):
    fake_data = {
        "dallas":   {"city": "Dallas",   "temperature": "82F", "condition": "Sunny"},
        "new york": {"city": "New York", "temperature": "61F", "condition": "Cloudy"},
        "seattle":  {"city": "Seattle",  "temperature": "55F", "condition": "Rainy"},
    }
    # .lower() makes the lookup case-insensitive ("Dallas" and "dallas" both work)
    return fake_data.get(city.lower(), {"city": city, "temperature": "N/A", "condition": "Unknown"})

# ── STEP 2: Tell the model what functions exist ─────────────────────────────
# "tools" describes each function in JSON format.
# The model reads this and knows: "I can call get_weather if I need weather info."
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The city name, e.g. Dallas"}
                },
                "required": ["city"]
            }
        }
    }
]

# ── STEP 3: Send the user question to the model ────────────────────────────
# The model sees the question AND the available tools.
# It decides on its own whether to answer directly or call a function.
messages = [{"role": "user", "content": "What is the weather like in Dallas today?"}]

response = client.chat.completions.create(
    model="gpt-4o-mini",  # Chat model required for function calling
    messages=messages,
    tools=tools,
    tool_choice="auto"    # "auto" = let the model decide when to use a tool
)

first_reply = response.choices[0].message

# The model does NOT answer yet — it asks us to run get_weather first
print("── Step 1: Model requests a function call ──")
print(f"Function : {first_reply.tool_calls[0].function.name}")
print(f"Arguments: {first_reply.tool_calls[0].function.arguments}")

# ── STEP 4: Run the function and send the result back ──────────────────────
args = json.loads(first_reply.tool_calls[0].function.arguments)
weather_result = get_weather(args["city"])

# Append the model's function-call request and our function result to the conversation
messages.append(first_reply)
messages.append({
    "role": "tool",
    "tool_call_id": first_reply.tool_calls[0].id,
    "content": json.dumps(weather_result)  # send data back as a JSON string
})

# ── STEP 5: Ask the model to generate a natural language answer ────────────
final_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

print("\n── Step 2: Model answers using the function result ──")
print(final_response.choices[0].message.content)

> **Try this:** Change `"Dallas"` in the user message to `"Seattle"` or `"New York"` and re-run. Then try a city not in the list (e.g., `"Austin"`) — what does the model say?
>
> **Reflection:** Here `get_weather` returns fake data. In a real app you would replace it with a live weather API call. What other real-world functions could you connect to an LLM this way?

---
# Part 2: Web App Basics

In Part 2, you will build a tiny interactive web page right inside this notebook. We use a Python trick: `IPython.display.HTML` renders HTML directly in Colab output — no browser needed.

Think of this as a preview of what your actual AI application does under the hood.

## 2.1 HTML — Structure

HTML defines the **structure** of a page — what elements exist and where. Run the cell below to see a simple form rendered.

In [1]:
from IPython.display import HTML

html_code = """
<html>
<body>
    <h2>Product Copy Generator</h2>
    <label>Product Name</label><br>
    <input type="text" placeholder="Water Bottle" id="name"><br><br>
    <button id="submit-btn">Generate Copy</button>
</body>
</html>
"""

HTML(html_code)

You can see a heading, a text input, and a button — but clicking the button does nothing yet. That is where JavaScript comes in.

---
## 2.2 Exercise: JavaScript — Behavior

JavaScript makes a page **interactive**. The code below listens for a button click and displays a message.

**Your task:** Fill in the blank so that clicking the button shows the message:  
`"Your product is: "` followed by whatever the user typed.

In [2]:
from IPython.display import HTML

html_with_js = """
<html>
<body>
    <h2>Product Copy Generator</h2>
    <label>Product Name</label><br>
    <input type="text" placeholder="Water Bottle" id="name"><br><br>
    <button id="submit-btn">Generate Copy</button>
    <p id="output"></p>

    <script>
        document.getElementById("submit-btn").addEventListener("click", () => {
            const productName = document.getElementById("name").value;
            // Fill in the blank: display "Your product is: " + productName in the <p> tag
            document.getElementById("output").innerText = ___;
        });
    </script>
</body>
</html>
"""

HTML(html_with_js)

> **Hint:** In JavaScript, you can combine text with a variable using a template literal:  
> `` `Your product is: ${productName}` ``

---
## 2.3 Exercise: CSS — Style

CSS controls how elements **look** — colors, fonts, spacing. The button below is unstyled. Your task: fill in the CSS so the button has a **blue background** and **white text**.

In [4]:
from IPython.display import HTML

html_with_css = """
<html>
<head>
    <style>
        body {
            font-family: Arial, sans-serif;
            padding: 20px;
        }
        button {
            background: ___;
            color: ___;
            padding: 8px 16px;
            border: none;
            border-radius: 4px;
            cursor: pointer;
        }
        button:hover {
            background: #59C3C3;
            transition: 0.7s;
        }
    </style>
</head>
<body>
    <h2>Product Copy Generator</h2>
    <label>Product Name</label><br>
    <input type="text" placeholder="Water Bottle" id="name"><br><br>
    <button id="submit-btn">Generate Copy</button>
    <p id="output"></p>

    <script>
        document.getElementById("submit-btn").addEventListener("click", () => {
            const productName = document.getElementById("name").value;
            document.getElementById("output").innerText = `Your product is: ${productName}`;
        });
    </script>
</body>
</html>
"""

HTML(html_with_css)